# curebench results
- dataset: curebench_valset_pharse1.jsonl

In [1]:
import os
import sys
parent_dir = os.path.split(os.getcwd())[0]
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
print(f"Adding directory\n{parent_dir}\nto sys.path")

Adding directory
/Users/christiembp/Documents/workspace/CUREBench
to sys.path


In [2]:
import os
import json
import pandas as pd
import glob
from torch.utils.data import DataLoader
from dataset_utils import build_dataset

In [3]:
# pandas settings
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### load val dataset

In [4]:
# annotated val data: added question categories
val_data_path = os.path.join(parent_dir, "resources/curebench_testset_phase1-with-categories.csv")
val_df = pd.read_csv(val_data_path)

In [5]:
print(f"number of rows in val set: {len(val_df):,}")

number of rows in val set: 459


In [6]:
val_df["question_index"] = val_df.index

In [7]:
val_df.head(1)

,question_type,id,question,answer,meta_question,task_type,reasoning_trace,question_index
0,multi_choice,U9PHZ83RKYV8,Which drug brand name is associated with the treatment of acne?\nA: Salicylic Acid\nB: Minoxidil\nC: Ketoconazole\nD: Fluocinonide,A,NaN,Drug Overview,"[{'role': 'assistant', 'channel': 'analysis', 'content': 'Recognize the question and map it to the most relevant Task Type. The question asks about the drug brand name used for acne treatment. This falls under basic drug knowledge regarding brand names and indications. This is a basic drug fact question, not directly matching any of the Task Type categories listed. Since none of the categories directly correspond, we need to identify which category is closest.\n\nThe options given involve basic drug facts about usage:\n\n- Salicylic Acid is used for acne.\n- Minoxidil is used for hair growth, not acne.\n- Ketoconazole is an antifungal, not primarily used for acne.\n- Fluocinonide is a corticosteroid, used for inflammatory skin conditions but not specifically acne.\n\nThe question is effectively about ""advice on typical drug usage"" which would best fall under the ""Drug Overview"" category. The term ""Drug Overview"" is the broad category covering general information about the drug.\n\nThus, the best match is ""Drug Overview"".'}, {'role': 'assistant', 'channel': 'final', 'content': 'Drug Overview'}]",0


In [8]:
task_types = [
  {
    "task_type": "Treatment Recommendation",
    "description": "Questions regarding specialized treatment recommendations considering patient populations."
  },
  {
    "task_type": "Adverse Event",
    "description": "Predicting potential adverse events and side effects based on drug properties and patient factors."
  },
  {
    "task_type": "Drug Overview",
    "description": "Package label principal display panel and comprehensive drug description."
  },
  {
    "task_type": "Drug Ingredients",
    "description": "Product data elements and active/inactive ingredient analysis."
  },
  {
    "task_type": "Drug Warnings and Safety",
    "description": "Boxed warnings, contraindications, adverse reactions, and drug interactions."
  },
  {
    "task_type": "Drug Dependence and Abuse",
    "description": "Drug abuse potential, dependence, controlled substance classification, and overdosage."
  },
  {
    "task_type": "Dosage and Administration",
    "description": "Indications, usage, administration, dosage forms, strengths, and instructions."
  },
  {
    "task_type": "Drug Use in Specific Populations",
    "description": "Usage in pregnancy, pediatric, geriatric populations, and nursing mothers."
  },
  {
    "task_type": "Pharmacology",
    "description": "Clinical pharmacology, mechanism of action, pharmacodynamics, and pharmacokinetics."
  },
  {
    "task_type": "Clinical Information",
    "description": "Clinical studies and trial data analysis."
  },
  {
    "task_type": "Nonclinical Toxicology",
    "description": "Toxicology, carcinogenesis, mutagenesis, fertility impairment, and animal studies."
  },
  {
    "task_type": "Patient-Focused Information",
    "description": "Patient medication guides, package inserts, and patient information."
  }
]

In [9]:
print(f"number of task types: {len(task_types)}")

number of task types: 12


## read in partial results and stich them together

In [10]:
results_path = os.path.join(parent_dir, "resources/curebench-val-results")
file_pattern = os.path.join(results_path, '*.csv')
all_files = glob.glob(file_pattern)
df_list = [pd.read_csv(f) for f in all_files]
results_df = pd.concat(df_list, ignore_index=True)

In [11]:
print(f"number of observations in combined results: {len(results_df)}")

number of observations in combined results: 459


In [12]:
all_files

['/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission.csv']

In [13]:
for i, x in enumerate(df_list):
    print(all_files[i], len(x))

/Users/christiembp/Documents/workspace/CUREBench/resources/curebench-val-results/submission.csv 459


In [14]:
results_df.drop_duplicates()
print(len(results_df))

459


In [15]:
results_df.head(2)

,id,prediction,choice,reasoning
0,U9PHZ83RKYV8,A,A,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The user asks: \""Which drug brand name is associated with the treatment of acne? A: Salicylic Acid B: Minoxidil C: Ketoconazole D: Fluocinonide\""\n\nWe need to determine if the question is multiple-choice. Yes, they provide choices A-D. So MC. We must answer with letter only. The correct choice is Salicylic Acid (commonly used for acne). So letter \""A\"".""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""A""}]"
1,vIGwm8qguXYi,"If a patient develops a severe allergic reaction during or after receiving fosaprepitant, they should immediately stop the infusion, notify the healthcare provider, and receive urgent medical care—ideally by contacting emergency services and receiving treatment with epinephrine, antihistamines, and corticosteroids as directed.",B,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""Open-ended. Need to answer succinctly.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""If a patient develops a severe allergic reaction during or after receiving fosaprepitant, they should immediately stop the infusion, notify the healthcare provider, and receive urgent medical care\u2014ideally by contacting emergency services and receiving treatment with epinephrine, antihistamines, and corticosteroids as directed.""}, {""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""We need to convert to final answer. They gave open-ended answer. The correct option is B.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""B""}]"


In [16]:
results_df = pd.merge(results_df, val_df, on='id', how='left')

In [17]:
question_frequencies = results_df['question_index'].value_counts()

In [18]:
question_frequencies[question_frequencies > 1].head()

Series([], Name: count, dtype: int64)

In [19]:
results_df[results_df.prediction=="Error"].head(2)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,task_type,reasoning_trace,question_index


#### drop errors and duplicates

In [20]:
non_error_df = results_df[results_df.prediction!="Error"].copy()

In [21]:
deduplicated_df = non_error_df.drop_duplicates(subset=['id'], keep='first')

In [22]:
len(deduplicated_df )

459

### read in combined results

In [23]:
combined_results = deduplicated_df.copy()

In [24]:
# combined_data_path = os.path.join(parent_dir, "resources/curebench-val-results/unsloth_gpt-oss-20b_finetuned_results_val.csv")
# combined_results = pd.read_csv(combined_data_path)

In [25]:
print(f"number of rows in combined results: {len(combined_results)}")

number of rows in combined results: 459


### compute results

In [24]:
combined_results.head(1)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,task_type,reasoning_trace,question_index
0,U9PHZ83RKYV8,A,A,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The user asks: \""Which drug brand name is associated with the treatment of acne? A: Salicylic Acid B: Minoxidil C: Ketoconazole D: Fluocinonide\""\n\nWe need to determine if the question is multiple-choice. Yes, they provide choices A-D. So MC. We must answer with letter only. The correct choice is Salicylic Acid (commonly used for acne). So letter \""A\"".""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""A""}]",multi_choice,Which drug brand name is associated with the treatment of acne?\nA: Salicylic Acid\nB: Minoxidil\nC: Ketoconazole\nD: Fluocinonide,A,NaN,Drug Overview,"[{'role': 'assistant', 'channel': 'analysis', 'content': 'Recognize the question and map it to the most relevant Task Type. The question asks about the drug brand name used for acne treatment. This falls under basic drug knowledge regarding brand names and indications. This is a basic drug fact question, not directly matching any of the Task Type categories listed. Since none of the categories directly correspond, we need to identify which category is closest.\n\nThe options given involve basic drug facts about usage:\n\n- Salicylic Acid is used for acne.\n- Minoxidil is used for hair growth, not acne.\n- Ketoconazole is an antifungal, not primarily used for acne.\n- Fluocinonide is a corticosteroid, used for inflammatory skin conditions but not specifically acne.\n\nThe question is effectively about ""advice on typical drug usage"" which would best fall under the ""Drug Overview"" category. The term ""Drug Overview"" is the broad category covering general information about the drug.\n\nThus, the best match is ""Drug Overview"".'}, {'role': 'assistant', 'channel': 'final', 'content': 'Drug Overview'}]",0


In [25]:
combined_results["correct"] = (combined_results['choice'] == combined_results['answer'])

In [26]:
combined_results.to_csv("unsloth_gpt-oss-20b_base_results_val.csv", index=False)

In [27]:
non_open_ended_df = combined_results[combined_results.question_type!="open_ended"]

In [28]:
len(non_open_ended_df[non_open_ended_df.correct==True])

263

In [29]:
# Calculate and log overall accuracy
total_correct = len(non_open_ended_df[non_open_ended_df.correct==True])
total_examples = len(non_open_ended_df)
overall_accuracy = total_correct / total_examples if total_examples > 0 else 0.0

print(f"Overall accuracy (excluding open-ended questions): {overall_accuracy:.2%} ({total_correct}/{total_examples})")

Overall accuracy (excluding open-ended questions): 63.68% (263/413)


### inspect output

In [30]:
correct_df = non_open_ended_df[non_open_ended_df.correct==True]

In [31]:
wrong_df = non_open_ended_df[non_open_ended_df.correct!=True]

In [32]:
wrong_df.head(1)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,task_type,reasoning_trace,question_index,correct
3,WfWiWK0yULaX,"Renal impairment (e.g., advanced kidney disease) is a contraindication for the use of Gadavist.",C,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The user: \""Which of the following conditions is a contraindication for the use of Gadavist?\"" This is a multiple-choice question likely with options. They didn't provide options here. Possibly the answer may be \""renal impairment\"" or \""renal failure\"". But we need to determine if question is MC or OE. Since it's asking: \""Which of the following conditions is a contraindication for the use of Gadavist?\"" They expect a letter. But options unknown. Can't give letter. Maybe answer: Renal impairment. But need to say 'renal impairment' as answer. Since no options, treat as OE. Provide a succinct single sentence response.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""Renal impairment (e.g., advanced kidney disease) is a contraindication for the use of Gadavist.""}, {""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""Need to output the letter: Option C.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""C""}]",open_ended_multi_choice,Which of the following conditions is a contraindication for the use of Gadavist?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: Which of the following conditions is a contraindication for the use of Gadavist?\nA: Mild hypersensitivity reactions to Gadavist\nB: History of severe hypersensitivity reactions to Gadavist\nC: Renal impairment\nD: Liver dysfunction\n\n",Drug Warnings and Safety,"[{'role': 'assistant', 'channel': 'analysis', 'content': 'The user is asking about contraindications for Gadavist. This is about drug use, specifically identifying a contraindication condition. According to the task types, the question asks for information about usage contraindications for a drug. The relevant category is ""Drug Warnings and Safety,"" which covers contraindications.'}, {'role': 'assistant', 'channel': 'commentary', 'content': 'The most relevant Task Type is ""Drug Warnings and Safety.""'}, {'role': 'assistant', 'channel': 'final', 'content': 'Drug Warnings and Safety'}]",3,False


In [33]:
# Extract just the task_type strings
valid_task_types = [t['task_type'] for t in task_types]

# Filter and then group
filtered_df = non_open_ended_df[non_open_ended_df['task_type'].isin(valid_task_types)]

counts = filtered_df.groupby(['task_type', 'correct']).size().unstack(fill_value=0)
percentages = counts.div(counts.sum(axis=1), axis=0) * 100

result = pd.DataFrame({
    'correct_pct': percentages[False].round(2),
    'incorrect_pct': percentages[True].round(2),
    'total_count': counts.sum(axis=1)
})

In [34]:
result

,correct_pct,incorrect_pct,total_count
task_type,,,
Adverse Event,16.67,83.33,60
Clinical Information,51.85,48.15,27
Dosage and Administration,58.18,41.82,55
Drug Dependence and Abuse,42.86,57.14,7
Drug Ingredients,0.00,100.00,5
Drug Overview,35.00,65.00,20
Drug Use in Specific Populations,36.76,63.24,68
Drug Warnings and Safety,36.36,63.64,66
Nonclinical Toxicology,33.33,66.67,9


In [35]:
correct_question_counts = correct_df.groupby(['question_type'], as_index=False).size() # get counts
correct_question_counts.sort_values('size', ascending=False, inplace=True) # sort by size

In [36]:
correct_question_counts

,question_type,size
0,multi_choice,133
1,open_ended_multi_choice,130


In [37]:
wrong_question_counts = wrong_df.groupby(['question_type'], as_index=False).size() # get counts
wrong_question_counts.sort_values('size', ascending=False, inplace=True) # sort by size

In [38]:
wrong_question_counts

,question_type,size
1,open_ended_multi_choice,100
0,multi_choice,50


In [39]:
wrong_df[wrong_df.question_type=="open_ended_multi_choice"].head(1)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,task_type,reasoning_trace,question_index,correct
3,WfWiWK0yULaX,"Renal impairment (e.g., advanced kidney disease) is a contraindication for the use of Gadavist.",C,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""The user: \""Which of the following conditions is a contraindication for the use of Gadavist?\"" This is a multiple-choice question likely with options. They didn't provide options here. Possibly the answer may be \""renal impairment\"" or \""renal failure\"". But we need to determine if question is MC or OE. Since it's asking: \""Which of the following conditions is a contraindication for the use of Gadavist?\"" They expect a letter. But options unknown. Can't give letter. Maybe answer: Renal impairment. But need to say 'renal impairment' as answer. Since no options, treat as OE. Provide a succinct single sentence response.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""Renal impairment (e.g., advanced kidney disease) is a contraindication for the use of Gadavist.""}, {""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""Need to output the letter: Option C.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""C""}]",open_ended_multi_choice,Which of the following conditions is a contraindication for the use of Gadavist?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: Which of the following conditions is a contraindication for the use of Gadavist?\nA: Mild hypersensitivity reactions to Gadavist\nB: History of severe hypersensitivity reactions to Gadavist\nC: Renal impairment\nD: Liver dysfunction\n\n",Drug Warnings and Safety,"[{'role': 'assistant', 'channel': 'analysis', 'content': 'The user is asking about contraindications for Gadavist. This is about drug use, specifically identifying a contraindication condition. According to the task types, the question asks for information about usage contraindications for a drug. The relevant category is ""Drug Warnings and Safety,"" which covers contraindications.'}, {'role': 'assistant', 'channel': 'commentary', 'content': 'The most relevant Task Type is ""Drug Warnings and Safety.""'}, {'role': 'assistant', 'channel': 'final', 'content': 'Drug Warnings and Safety'}]",3,False


In [40]:
wrong_df[wrong_df.question_type=="open_ended_multi_choice"].groupby(['choice'], as_index=False).size()

,choice,size
0,A,18
1,B,11
2,C,13
3,D,11
4,NOTAVALUE,47


In [41]:
wrong_df[wrong_df.task_type=="Nonclinical Toxicology"].head(2)

,id,prediction,choice,reasoning,question_type,question,answer,meta_question,task_type,reasoning_trace,question_index,correct
23,Eic83tab2ldk,C,C,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""Need answer. Likely study: oral dose of 30 mg/kg? But I'm not sure. Maybe 0 mg? They found no evidence of impaired fertility at doses up to 500 mg/kg? Quick search memory: \""Albuterol sulfate oral oral\"". Probably no evidence up to 100 mg/kg. I'll guess 30 mg/kg.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""C""}, {""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""We just need output \""C\"".""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""C""}]",open_ended_multi_choice,At what oral dose did albuterol sulfate demonstrate no evidence of impaired fertility in rats?,B,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: At what oral dose did albuterol sulfate demonstrate no evidence of impaired fertility in rats?\nA: 2 mg/kg\nB: 50 mg/kg\nC: 500 mg/kg\nD: 100 mg/kg\n\n",Nonclinical Toxicology,"[{'role': 'assistant', 'channel': 'analysis', 'content': 'The question is about which oral dose of albuterol sulfate did not show evidence of impaired fertility in rats. This pertains to findings from a toxicological study about reproductive toxicity, specifically nonclinical toxicology effects. None of the other categories directly address a dose-based outcome from a rat fertility test. So the correct classification is Nonclinical Toxicology.'}, {'role': 'assistant', 'channel': 'final', 'content': 'Nonclinical Toxicology'}]",23,False
51,ZSuguD8bB8HX,Studies that assess mutagenicity of Hydroxychloroquine (Hydro) are typically carried out as **in vitro assays** using cell lines such as the Ames test or mammalian cell mutagenicity tests.,D,"[{""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""Need to answer: It's an open-ended question? Probably ask which category of studies. Provide short answer.""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""Studies that assess mutagenicity of Hydroxychloroquine (Hydro) are typically carried out as **in vitro assays** using cell lines such as the Ames test or mammalian cell mutagenicity tests.""}, {""role"": ""assistant"", ""channel"": ""analysis"", ""content"": ""We need to output the correct letter. The correct answer: category of studies has been conducted to determine mutagenicity for Hydrocort? None of the options match the description. The correct likely is D? Actually the agent answer is wrong (talking about hydroxychloroquine). The correct answer likely is D: Studies with topical corticosteroids in humans. But not sure. However the instruction: output the letter. So answer \""D\"".""}, {""role"": ""assistant"", ""channel"": ""final"", ""content"": ""D""}]",open_ended_multi_choice,Which category of studies has been conducted to determine mutagenicity for Hydrocort?,A,"The following is a multiple choice question about medicine and the agent's open-ended answer to the question. Convert the agent's answer to the final answer format using the corresponding option label, e.g., 'A', 'B', 'C', 'D', 'E' or 'None'. \n\nQuestion: Which category of studies has been conducted to determine mutagenicity for Hydrocort?\nA: Studies with prednisolone and hydrocortisone\nB: Studies with dexamethasone and betamethasone\nC: Studies with triamcinolone and fluticasone\nD: Studies with topical corticosteroids in humans\n\n",Nonclinical Toxicology,"[{'role': 'assistant', 'channel': 'analysis', 'content': 'Need to identify category: mutagenicity studies: this falls under Nonclinical Toxicology.'}, {'role': 'assistant', 'channel': 'final', 'content': 'Nonclinical Toxicology'}]",51,False
